# P4 Strict: Grade Signal Outcome Confirmatory Models

RAW_A와 P3-S OOF residual이 건강보험 취업률 및 대학원 진학률에 제공하는 추가정보를 비교한다.
P3 residual은 upstream artifact로 읽고, P4-Q는 현행 P2-Q approval gate 때문에 실행하지 않는다.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts/p3_p4_strict_pipeline_lib.py").exists():
    PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import p3_p4_strict_pipeline_lib as lib

OUTPUT_ROOT = lib.P4_ROOT
NOTEBOOK_PATH = OUTPUT_ROOT / "p4_grade_signal_outcomes_strict.ipynb"
lib.ensure_dirs(OUTPUT_ROOT)
ENV = lib.execution_environment(NOTEBOOK_PATH, OUTPUT_ROOT)
display(pd.DataFrame([ENV]))

,python,platform,pandas,numpy,statsmodels,working_directory,git_commit,execution_timestamp_utc,notebook_path,output_root
0,3.12.3,Linux-6.18.33.2-microsoft-standard-WSL2-x86_64...,3.0.3,2.5.0,0.14.6,/home/sieg/projects-wsl/SBS_dataScience,5b1a3d54266d881a839ad9a3cec750da66e94bc7,2026-07-13T07:28:29.228482+00:00,/home/sieg/projects-wsl/SBS_dataScience/workbo...,/home/sieg/projects-wsl/SBS_dataScience/workbo...


In [2]:
df_d08, df_membership, df_registry, p2_status, p2_handoff, INPUTS = lib.load_base_inputs()
df_base = lib.join_membership(df_d08, df_membership)
p3_full_path = lib.P3_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_FULL.parquet"
p3_nopolicy_path = lib.P3_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_NOPOLICY.parquet"
p3_status_path = lib.P3_ROOT / "reports/P3_GRADE_RESIDUAL_STATUS.json"
if not p3_full_path.exists():
    raise FileNotFoundError(p3_full_path)
p3_full = pd.read_parquet(p3_full_path)
p3_nopolicy = pd.read_parquet(p3_nopolicy_path)
p3_status = json.loads(p3_status_path.read_text(encoding="utf-8"))
input_audit = pd.DataFrame([
    {"name": k, "path": lib.rel(v), "exists": v.exists(), "shape": lib.file_shape(v), "sha256": lib.sha256_file(v)}
    for k, v in {**INPUTS, "p3_full_residual": p3_full_path, "p3_nopolicy_residual": p3_nopolicy_path}.items()
])
input_audit.to_csv(OUTPUT_ROOT / "qa/P4_INPUT_CONTRACT_AUDIT.csv", index=False)
display(input_audit)

,name,path,exists,shape,sha256
0,strict_d08,workbook/p2/p2_4/source_eda/strict_clean_v1/ma...,True,"[7592, 151]",5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...
1,sample_membership,workbook/p2/p2_4/p4_modeling_readiness_v4/data...,True,"[7592, 21]",c6504fd04c918ce33439462f7b277aa8ae4363f185ef93...
2,sample_registry,workbook/p2/p2_4/p4_modeling_readiness_v4/arti...,True,"[12, 12]",d518bab363399a31037590a2cb285f2a76e88318f5d25b...
3,manual_feature_registry,workbook/p2/p2_4/source_eda/human_handoff_pack...,True,"[198, 11]",2cdb7797c4619c625fc6a171710970b7691446f4d62cae...
4,target_denylist,workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P...,True,"[16, 3]",97435cb3a6b20b490ce2ce7e65160414517aee5b472f8e...
5,p2_status,workbook/p2/p2_4/p2_grade_formation_v1_strict/...,True,[30],fa48e5d973cec5868a8983e424f56e6fc5d2d999a072a6...
6,p2_handoff,workbook/p2/p2_4/p2_grade_formation_v1_strict/...,True,[11],2bbc7d64784b7b530d57bb6a2096d14cb11c5879f41a52...
7,strict_target_membership,workbook/p2/p2_4/source_eda/strict_clean_v1/st...,True,"[61452, 7]",29bf19120774e7d0a86e1c6b6892d012832f3fd6fd5dbe...
8,strict_target_counts,workbook/p2/p2_4/source_eda/strict_clean_v1/st...,True,"[6, 7]",dde484367e1849b34a28b17e99381f59c04c94fb549d78...
9,p3_full_residual,workbook/p2/p2_4/p3_grade_residual_v1_strict/d...,True,"[7592, 10]",d8decd39dca42ccd0dc194fa3813ba0036541098eea08d...


In [3]:
emp = lib.build_p4_frame(df_base, p3_full.merge(p3_nopolicy[[lib.KEY, "grade_residual_structure_nopolicy_oof_pp", "grade_residual_structure_nopolicy_oof_10pp"]], on=lib.KEY), "sample_P4_E_STRUCTURE_READY", "health_employment_rate_pct")
prog = lib.build_p4_frame(df_base, p3_full.merge(p3_nopolicy[[lib.KEY, "grade_residual_structure_nopolicy_oof_pp", "grade_residual_structure_nopolicy_oof_10pp"]], on=lib.KEY), "sample_P4_P_STRUCTURE_READY", "graduate_school_progression_rate_pct")
joint = lib.build_p4_frame(df_base, p3_full.merge(p3_nopolicy[[lib.KEY, "grade_residual_structure_nopolicy_oof_pp", "grade_residual_structure_nopolicy_oof_10pp"]], on=lib.KEY), "sample_P4_JOINT_STRUCTURE_READY", "health_employment_rate_pct")
joint["graduate_school_progression_rate_prop"] = pd.to_numeric(joint["graduate_school_progression_rate_pct"], errors="coerce") / 100.0
emp.to_parquet(OUTPUT_ROOT / "data/P4_STRUCTURE_EMPLOYMENT_FRAME.parquet", index=False)
prog.to_parquet(OUTPUT_ROOT / "data/P4_STRUCTURE_PROGRESSION_FRAME.parquet", index=False)
joint.to_parquet(OUTPUT_ROOT / "data/P4_STRUCTURE_JOINT_FRAME.parquet", index=False)
sample_audit = pd.DataFrame([
    lib.sample_audit(df_base, "sample_P4_E_STRUCTURE_READY", df_registry, "P4_E_STRUCTURE_READY", "health_employment_rate_pct"),
    lib.sample_audit(df_base, "sample_P4_P_STRUCTURE_READY", df_registry, "P4_P_STRUCTURE_READY", "graduate_school_progression_rate_pct"),
    lib.sample_audit(df_base, "sample_P4_JOINT_STRUCTURE_READY", df_registry, "P4_JOINT_STRUCTURE_READY", "health_employment_rate_pct"),
])
sample_audit.to_csv(OUTPUT_ROOT / "qa/P4_SAMPLE_AUDIT.csv", index=False)
display(sample_audit)

,sample_id,row_n,registry_row_n,school_n,registry_school_n,train_n,registry_train_n,validation_n,registry_validation_n,test_n,registry_test_n,target_non_null_n,row_id_hash
0,P4_E_STRUCTURE_READY,5600,5600,185,185,4080,4080,871,871,649,649,5600,3f7f76bc4e1a9680052b996b012021d36ccf3374fa182f...
1,P4_P_STRUCTURE_READY,5674,5674,194,194,4129,4129,884,884,661,661,5674,6625f4fcb391ac2e7d9a3001be72425a2dd4f0f6e03ef4...
2,P4_JOINT_STRUCTURE_READY,5600,5600,185,185,4080,4080,871,871,649,649,5600,3f7f76bc4e1a9680052b996b012021d36ccf3374fa182f...


In [4]:
target_profile = []
for name, frame, col in [
    ("HEALTH_EMPLOYMENT", emp, "health_employment_rate_pct"),
    ("GRAD_SCHOOL_PROGRESSION", prog, "graduate_school_progression_rate_pct"),
]:
    s = pd.to_numeric(frame[col], errors="coerce")
    target_profile.append({
        "outcome": name, "n": int(s.notna().sum()), "mean": float(s.mean()), "std": float(s.std()),
        "min": float(s.min()), "p05": float(s.quantile(.05)), "median": float(s.median()),
        "p95": float(s.quantile(.95)), "max": float(s.max()), "zero_ratio": float((s == 0).mean())
    })
target_profile = pd.DataFrame(target_profile)
target_profile.to_csv(OUTPUT_ROOT / "artifacts/P4_TARGET_PROFILE.csv", index=False)
display(target_profile)

,outcome,n,mean,std,min,p05,median,p95,max,zero_ratio
0,HEALTH_EMPLOYMENT,5600,52.694103,17.51351,0.0,23.076923,52.941177,82.223260,100.0,0.003571
1,GRAD_SCHOOL_PROGRESSION,5674,8.254616,12.89555,0.0,0.000000,2.777778,35.778388,100.0,0.412760


In [5]:
signal_cols = {
    "RAW_A": "a_rate_10pp",
    "OOF_RESIDUAL_FULL": "grade_residual_structure_full_oof_10pp",
    "OOF_RESIDUAL_NOPOLICY": "grade_residual_structure_nopolicy_oof_10pp",
}
coverage_rows = []
for frame_name, frame in [("employment", emp), ("progression", prog), ("joint", joint)]:
    for label, col in signal_cols.items():
        coverage_rows.append({"frame": frame_name, "grade_signal": label, "column": col, "row_n": len(frame), "non_null_n": int(frame[col].notna().sum()), "coverage_rate": float(frame[col].notna().mean())})
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(OUTPUT_ROOT / "qa/P4_SIGNAL_COVERAGE_AUDIT.csv", index=False)
display(coverage)

,frame,grade_signal,column,row_n,non_null_n,coverage_rate
0,employment,RAW_A,a_rate_10pp,5600,5600,1.0
1,employment,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,5600,5600,1.0
2,employment,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,5600,5600,1.0
3,progression,RAW_A,a_rate_10pp,5674,5674,1.0
4,progression,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,5674,5674,1.0
5,progression,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,5674,5674,1.0
6,joint,RAW_A,a_rate_10pp,5600,5600,1.0
7,joint,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,5600,5600,1.0
8,joint,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,5600,5600,1.0


In [6]:
cat = list(p2_handoff["p2_s_final_spec"]["categorical"])
num = list(p2_handoff["p2_s_final_spec"]["numeric"])
grade_cat = cat
grade_num = num
model_specs = pd.DataFrame(
    [{"outcome": o, "grade_signal": s, "model_family": f, "controls": "S0+B_CORE+B_SCALE+Policy"}
     for o in ["HEALTH_EMPLOYMENT", "GRAD_SCHOOL_PROGRESSION"]
     for s in signal_cols
     for f in ["fractional_logit", "ols_cluster"]]
)
model_specs.to_csv(OUTPUT_ROOT / "artifacts/P4_MODEL_SPEC_REGISTRY.csv", index=False)
display(model_specs)

,outcome,grade_signal,model_family,controls
0,HEALTH_EMPLOYMENT,RAW_A,fractional_logit,S0+B_CORE+B_SCALE+Policy
1,HEALTH_EMPLOYMENT,RAW_A,ols_cluster,S0+B_CORE+B_SCALE+Policy
2,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,fractional_logit,S0+B_CORE+B_SCALE+Policy
3,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,ols_cluster,S0+B_CORE+B_SCALE+Policy
4,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,fractional_logit,S0+B_CORE+B_SCALE+Policy
5,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,ols_cluster,S0+B_CORE+B_SCALE+Policy
6,GRAD_SCHOOL_PROGRESSION,RAW_A,fractional_logit,S0+B_CORE+B_SCALE+Policy
7,GRAD_SCHOOL_PROGRESSION,RAW_A,ols_cluster,S0+B_CORE+B_SCALE+Policy
8,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,fractional_logit,S0+B_CORE+B_SCALE+Policy
9,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,ols_cluster,S0+B_CORE+B_SCALE+Policy


In [7]:
print("fractional logit helper functions loaded from shared library")

fractional logit helper functions loaded from shared library


In [8]:
emp_results, emp_ame, emp_test = lib.p4_signal_models(emp, "health_employment_rate_pct", "HEALTH_EMPLOYMENT", cat, num, signal_cols)
display(emp_results)

,model_id,outcome,outcome_pct,sample_id,grade_signal,grade_signal_column,model_family,N,school_n,beta,...,iv_deviance,iv_brier,iv_mae,test_deviance,test_brier,test_mae,converged,rank,status,warning
0,HEALTH_EMPLOYMENT_RAW_A_fractional_logit,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,fractional_logit,4951,158,0.025553,...,0.000225,0.000058,0.000158,1.362097,0.027230,0.125970,True,37,READY,
1,HEALTH_EMPLOYMENT_RAW_A_ols_cluster,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,ols_cluster,4951,158,0.006195,...,0.000097,0.000033,0.000084,1.362244,0.027251,0.125917,True,37,READY,
2,HEALTH_EMPLOYMENT_OOF_RESIDUAL_FULL_fractional...,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,fractional_logit,4951,158,0.026359,...,0.000262,0.000068,0.000173,1.360693,0.026899,0.125346,True,37,READY,
3,HEALTH_EMPLOYMENT_OOF_RESIDUAL_FULL_ols_cluster,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,ols_cluster,4951,158,0.006381,...,0.000128,0.000042,0.000096,1.360774,0.026910,0.125294,True,37,READY,
4,HEALTH_EMPLOYMENT_OOF_RESIDUAL_NOPOLICY_fracti...,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,fractional_logit,4951,158,0.025707,...,0.000244,0.000063,0.000169,1.360661,0.026892,0.125341,True,37,READY,
5,HEALTH_EMPLOYMENT_OOF_RESIDUAL_NOPOLICY_ols_cl...,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,ols_cluster,4951,158,0.006223,...,0.000112,0.000038,0.000093,1.360742,0.026903,0.125288,True,37,READY,


In [9]:
prog_results, prog_ame, prog_test = lib.p4_signal_models(prog, "graduate_school_progression_rate_pct", "GRAD_SCHOOL_PROGRESSION", cat, num, signal_cols)
display(prog_results)

,model_id,outcome,outcome_pct,sample_id,grade_signal,grade_signal_column,model_family,N,school_n,beta,...,iv_deviance,iv_brier,iv_mae,test_deviance,test_brier,test_mae,converged,rank,status,warning
0,GRAD_SCHOOL_PROGRESSION_RAW_A_fractional_logit,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,fractional_logit,5013,165,0.228680,...,0.005394,0.000666,0.002048,0.566943,0.027118,0.101346,True,39,READY,
1,GRAD_SCHOOL_PROGRESSION_RAW_A_ols_cluster,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,ols_cluster,5013,165,0.016772,...,-0.039304,0.000043,-0.000980,0.615598,0.031226,0.110550,True,39,READY,
2,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_FULL_frac...,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,fractional_logit,5013,165,0.220272,...,0.005411,0.000669,0.002103,0.542177,0.021416,0.093240,True,39,READY,
3,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_FULL_ols_...,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,ols_cluster,5013,165,0.016095,...,-0.039177,0.000046,-0.000960,0.607058,0.029119,0.108123,True,39,READY,
4,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_NOPOLICY_...,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,fractional_logit,5013,165,0.221069,...,0.005302,0.000656,0.002066,0.542244,0.021445,0.093279,True,39,READY,
5,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_NOPOLICY_...,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,ols_cluster,5013,165,0.016049,...,-0.039299,0.000035,-0.000998,0.607094,0.029138,0.108153,True,39,READY,


In [10]:
coef = pd.concat([emp_results, prog_results], ignore_index=True)
primary_mask = coef["model_family"].eq("fractional_logit") & coef["status"].eq("READY")
coef.loc[primary_mask, "p_holm"] = lib.holm_adjust(coef.loc[primary_mask, "p_value"].tolist())
coef.to_csv(OUTPUT_ROOT / "artifacts/P4_COEFFICIENT_RESULTS.csv", index=False)
ame = pd.concat([emp_ame, prog_ame], ignore_index=True)
ame.to_csv(OUTPUT_ROOT / "artifacts/P4_AME_RESULTS.csv", index=False)
test_metrics = pd.concat([emp_test, prog_test], ignore_index=True)
test_metrics.to_csv(OUTPUT_ROOT / "artifacts/P4_LOCKED_TEST_METRICS.csv", index=False)
display(coef)

,model_id,outcome,outcome_pct,sample_id,grade_signal,grade_signal_column,model_family,N,school_n,beta,...,iv_brier,iv_mae,test_deviance,test_brier,test_mae,converged,rank,status,warning,p_holm
0,HEALTH_EMPLOYMENT_RAW_A_fractional_logit,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,fractional_logit,4951,158,0.025553,...,0.000058,0.000158,1.362097,0.027230,0.125970,True,37,READY,,7.281490e-02
1,HEALTH_EMPLOYMENT_RAW_A_ols_cluster,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,ols_cluster,4951,158,0.006195,...,0.000033,0.000084,1.362244,0.027251,0.125917,True,37,READY,,NaN
2,HEALTH_EMPLOYMENT_OOF_RESIDUAL_FULL_fractional...,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,fractional_logit,4951,158,0.026359,...,0.000068,0.000173,1.360693,0.026899,0.125346,True,37,READY,,7.281490e-02
3,HEALTH_EMPLOYMENT_OOF_RESIDUAL_FULL_ols_cluster,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,ols_cluster,4951,158,0.006381,...,0.000042,0.000096,1.360774,0.026910,0.125294,True,37,READY,,NaN
4,HEALTH_EMPLOYMENT_OOF_RESIDUAL_NOPOLICY_fracti...,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,fractional_logit,4951,158,0.025707,...,0.000063,0.000169,1.360661,0.026892,0.125341,True,37,READY,,7.281490e-02
5,HEALTH_EMPLOYMENT_OOF_RESIDUAL_NOPOLICY_ols_cl...,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,ols_cluster,4951,158,0.006223,...,0.000038,0.000093,1.360742,0.026903,0.125288,True,37,READY,,NaN
6,GRAD_SCHOOL_PROGRESSION_RAW_A_fractional_logit,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,fractional_logit,5013,165,0.228680,...,0.000666,0.002048,0.566943,0.027118,0.101346,True,39,READY,,3.041148e-11
7,GRAD_SCHOOL_PROGRESSION_RAW_A_ols_cluster,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,ols_cluster,5013,165,0.016772,...,0.000043,-0.000980,0.615598,0.031226,0.110550,True,39,READY,,NaN
8,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_FULL_frac...,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,fractional_logit,5013,165,0.220272,...,0.000669,0.002103,0.542177,0.021416,0.093240,True,39,READY,,2.630054e-11
9,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_FULL_ols_...,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,ols_cluster,5013,165,0.016095,...,0.000046,-0.000960,0.607058,0.029119,0.108123,True,39,READY,,NaN


In [11]:
ols_sensitivity = coef[coef["model_family"].eq("ols_cluster")].copy()
ols_sensitivity.to_csv(OUTPUT_ROOT / "artifacts/P4_OLS_SENSITIVITY.csv", index=False)
display(ols_sensitivity)

,model_id,outcome,outcome_pct,sample_id,grade_signal,grade_signal_column,model_family,N,school_n,beta,...,iv_brier,iv_mae,test_deviance,test_brier,test_mae,converged,rank,status,warning,p_holm
1,HEALTH_EMPLOYMENT_RAW_A_ols_cluster,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,ols_cluster,4951,158,0.006195,...,0.000033,0.000084,1.362244,0.027251,0.125917,True,37,READY,,NaN
3,HEALTH_EMPLOYMENT_OOF_RESIDUAL_FULL_ols_cluster,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,ols_cluster,4951,158,0.006381,...,0.000042,0.000096,1.360774,0.026910,0.125294,True,37,READY,,NaN
5,HEALTH_EMPLOYMENT_OOF_RESIDUAL_NOPOLICY_ols_cl...,HEALTH_EMPLOYMENT,health_employment_rate_pct,STRUCTURE,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,ols_cluster,4951,158,0.006223,...,0.000038,0.000093,1.360742,0.026903,0.125288,True,37,READY,,NaN
7,GRAD_SCHOOL_PROGRESSION_RAW_A_ols_cluster,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,RAW_A,a_rate_10pp,ols_cluster,5013,165,0.016772,...,0.000043,-0.000980,0.615598,0.031226,0.110550,True,39,READY,,NaN
9,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_FULL_ols_...,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,OOF_RESIDUAL_FULL,grade_residual_structure_full_oof_10pp,ols_cluster,5013,165,0.016095,...,0.000046,-0.000960,0.607058,0.029119,0.108123,True,39,READY,,NaN
11,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_NOPOLICY_...,GRAD_SCHOOL_PROGRESSION,graduate_school_progression_rate_pct,STRUCTURE,OOF_RESIDUAL_NOPOLICY,grade_residual_structure_nopolicy_oof_10pp,ols_cluster,5013,165,0.016049,...,0.000035,-0.000998,0.607094,0.029138,0.108153,True,39,READY,,NaN


In [12]:
# Progression is a fractional rate with many zeros but still continuous; this table records a two-part gate for transparency.
two_part = prog.assign(is_zero=lambda d: pd.to_numeric(d["graduate_school_progression_rate_pct"], errors="coerce").eq(0).astype(int))
two_part_profile = two_part.groupby("major_group_7")["is_zero"].agg(["count", "mean"]).reset_index()
two_part_profile.to_csv(OUTPUT_ROOT / "qa/P4_PROGRESSION_TWO_PART_PROFILE.csv", index=False)
display(two_part_profile)

,major_group_7,count,mean
0,ART,790,0.411392
1,EDU,640,0.678125
2,ENG,1282,0.255850
3,HUM,722,0.365651
4,MED,392,0.691327
5,NAT,720,0.123611
6,SOC,1128,0.559397


In [13]:
e2e_rows = []
for frame, outcome_pct, outcome_name in [
    (emp, "health_employment_rate_pct", "HEALTH_EMPLOYMENT"),
    (prog, "graduate_school_progression_rate_pct", "GRAD_SCHOOL_PROGRESSION"),
]:
    e2e = lib.group_cv_p4(frame, outcome_pct, outcome_name, cat, num, {"RAW_A": "a_rate_10pp", "OOF_RESIDUAL_FULL": "grade_residual_structure_full_oof_10pp"}, grade_cat, grade_num)
    e2e_rows.append(e2e)
e2e = pd.concat(e2e_rows, ignore_index=True)
e2e.to_csv(OUTPUT_ROOT / "qa/P4_END_TO_END_FOLD_AUDIT.csv", index=False)
incremental = e2e.groupby(["outcome", "grade_signal"]).agg(
    cv_deviance=("signal_deviance", "mean"),
    cv_brier=("signal_brier", "mean"),
    cv_mae=("signal_mae", "mean"),
    iv_deviance=("iv_deviance", "mean"),
    iv_brier=("iv_brier", "mean"),
    iv_mae=("iv_mae", "mean"),
).reset_index()
incremental.to_csv(OUTPUT_ROOT / "artifacts/P4_INCREMENTAL_VALUE.csv", index=False)
display(incremental)

,outcome,grade_signal,cv_deviance,cv_brier,cv_mae,iv_deviance,iv_brier,iv_mae
0,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,0.526522,0.012743,0.070314,0.006297,0.000660,0.001962
1,GRAD_SCHOOL_PROGRESSION,RAW_A,0.526522,0.012743,0.070314,0.006297,0.000660,0.001962
2,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,1.357162,0.023979,0.120475,0.000471,0.000129,0.000352
3,HEALTH_EMPLOYMENT,RAW_A,1.357162,0.023979,0.120475,0.000471,0.000129,0.000352


In [14]:
display(test_metrics)

,outcome,grade_signal,model_family,test_n,test_deviance,test_brier,test_mae,base_test_deviance,base_test_brier,base_test_mae
0,HEALTH_EMPLOYMENT,RAW_A,fractional_logit,649,1.362097,0.027230,0.125970,1.360279,0.026793,0.125622
1,HEALTH_EMPLOYMENT,RAW_A,ols_cluster,649,1.362244,0.027251,0.125917,1.360279,0.026793,0.125622
2,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,fractional_logit,649,1.360693,0.026899,0.125346,1.360279,0.026793,0.125622
3,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,ols_cluster,649,1.360774,0.026910,0.125294,1.360279,0.026793,0.125622
4,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,fractional_logit,649,1.360661,0.026892,0.125341,1.360279,0.026793,0.125622
5,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,ols_cluster,649,1.360742,0.026903,0.125288,1.360279,0.026793,0.125622
6,GRAD_SCHOOL_PROGRESSION,RAW_A,fractional_logit,661,0.566943,0.027118,0.101346,0.571045,0.028035,0.103206
7,GRAD_SCHOOL_PROGRESSION,RAW_A,ols_cluster,661,0.615598,0.031226,0.110550,0.571045,0.028035,0.103206
8,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,fractional_logit,661,0.542177,0.021416,0.093240,0.571045,0.028035,0.103206
9,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,ols_cluster,661,0.607058,0.029119,0.108123,0.571045,0.028035,0.103206


In [15]:
pilot = lib.fast_bootstrap_distribution(joint, cat, num, {"RAW_A": "a_rate_10pp", "OOF_RESIDUAL_FULL": "grade_residual_structure_full_oof_10pp"}, grade_cat, grade_num, iterations=100, seed=lib.RANDOM_STATE)
pilot.to_parquet(OUTPUT_ROOT / "artifacts/P4_BOOTSTRAP_PILOT_DISTRIBUTION.parquet", index=False)
display(pilot.groupby(["grade_signal", "status"]).size().reset_index(name="n"))

,grade_signal,status,n
0,OOF_RESIDUAL_FULL,OK,200
1,RAW_A,OK,200


In [16]:
boot = lib.fast_bootstrap_distribution(joint, cat, num, {"RAW_A": "a_rate_10pp", "OOF_RESIDUAL_FULL": "grade_residual_structure_full_oof_10pp"}, grade_cat, grade_num, iterations=500, seed=lib.RANDOM_STATE + 1)
boot.to_parquet(OUTPUT_ROOT / "artifacts/P4_BOOTSTRAP_DISTRIBUTION.parquet", index=False)
boot_ci = lib.bootstrap_ci(boot)
boot_ci.to_csv(OUTPUT_ROOT / "artifacts/P4_BOOTSTRAP_CI.csv", index=False)
display(boot_ci)

,outcome,grade_signal,metric,successful_n,ci_low,ci_high,median
0,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,ame,500,0.010073,0.024457,0.016740
1,GRAD_SCHOOL_PROGRESSION,RAW_A,ame,500,0.010073,0.024457,0.016740
2,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,ame,500,-0.000431,0.011780,0.005860
3,HEALTH_EMPLOYMENT,RAW_A,ame,500,-0.000431,0.011780,0.005860
4,PROGRESSION_MINUS_EMPLOYMENT,OOF_RESIDUAL_FULL,D,500,0.002916,0.018298,0.010658
5,PROGRESSION_MINUS_EMPLOYMENT,RAW_A,D,500,0.002916,0.018298,0.010658


In [17]:
primary = coef[coef["model_family"].eq("fractional_logit") & coef["status"].eq("READY")].copy()
wide = primary.pivot_table(index="grade_signal", columns="outcome", values="ame_10pp", aggfunc="first").reset_index()
if {"GRAD_SCHOOL_PROGRESSION", "HEALTH_EMPLOYMENT"}.issubset(wide.columns):
    wide["D_progression_minus_employment"] = wide["GRAD_SCHOOL_PROGRESSION"] - wide["HEALTH_EMPLOYMENT"]
wide.to_csv(OUTPUT_ROOT / "artifacts/P4_EMPLOYMENT_PROGRESSION_DIFFERENCE.csv", index=False)
display(wide)

outcome,grade_signal,GRAD_SCHOOL_PROGRESSION,HEALTH_EMPLOYMENT,D_progression_minus_employment
0,OOF_RESIDUAL_FULL,0.016577,0.006357,0.010220
1,OOF_RESIDUAL_NOPOLICY,0.016645,0.006200,0.010445
2,RAW_A,0.017260,0.006163,0.011097


In [18]:
mt = coef[["outcome", "grade_signal", "model_family", "p_value", "p_holm"]].copy()
mt.to_csv(OUTPUT_ROOT / "qa/P4_MULTIPLE_TEST_CORRECTION.csv", index=False)
display(mt)

,outcome,grade_signal,model_family,p_value,p_holm
0,HEALTH_EMPLOYMENT,RAW_A,fractional_logit,3.331334e-02,7.281490e-02
1,HEALTH_EMPLOYMENT,RAW_A,ols_cluster,3.328883e-02,NaN
2,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,fractional_logit,2.427163e-02,7.281490e-02
3,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,ols_cluster,2.446721e-02,NaN
4,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,fractional_logit,2.866542e-02,7.281490e-02
5,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,ols_cluster,2.886585e-02,NaN
6,GRAD_SCHOOL_PROGRESSION,RAW_A,fractional_logit,6.082297e-12,3.041148e-11
7,GRAD_SCHOOL_PROGRESSION,RAW_A,ols_cluster,1.405789e-06,NaN
8,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,fractional_logit,4.383423e-12,2.630054e-11
9,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,ols_cluster,9.229524e-07,NaN


In [19]:
# RAW_A and residual can become algebraically equivalent in linear outcome models
# when the same P3 grade controls are also present in P4. This audit makes that
# lineage constraint explicit instead of treating identical IV/bootstrap values as a surprise.
e2e_wide = e2e.pivot_table(index=["fold", "outcome"], columns="grade_signal", values=["iv_deviance", "iv_brier", "iv_mae"], aggfunc="first")
equivalence_rows = []
for metric in ["iv_deviance", "iv_brier", "iv_mae"]:
    if (metric, "RAW_A") in e2e_wide.columns and (metric, "OOF_RESIDUAL_FULL") in e2e_wide.columns:
        diff = (e2e_wide[(metric, "RAW_A")] - e2e_wide[(metric, "OOF_RESIDUAL_FULL")]).abs()
        equivalence_rows.append({"source": "end_to_end_cv", "metric": metric, "max_abs_raw_minus_residual": float(diff.max()), "allclose": bool(np.allclose(diff.fillna(0), 0, atol=1e-10))})
boot_wide = boot.pivot_table(index=["iteration", "outcome"], columns="grade_signal", values="ame", aggfunc="first")
if {"RAW_A", "OOF_RESIDUAL_FULL"}.issubset(boot_wide.columns):
    diff = (boot_wide["RAW_A"] - boot_wide["OOF_RESIDUAL_FULL"]).abs()
    equivalence_rows.append({"source": "bootstrap_fast_generated_regressor", "metric": "ame", "max_abs_raw_minus_residual": float(diff.max()), "allclose": bool(np.allclose(diff.fillna(0), 0, atol=1e-8))})
equivalence = pd.DataFrame(equivalence_rows)
equivalence["reason"] = "Residual equals raw A minus fitted P3 controls; with the same controls in a linear P4 design, RAW_A and residual span the same one-dimensional added information."
equivalence.to_csv(OUTPUT_ROOT / "qa/P4_WITHIN_RAW_EQUIVALENCE_AUDIT.csv", index=False)

diagnostics = coef[["model_id", "outcome", "grade_signal", "model_family", "status", "converged", "warning", "rank"]].copy()
diagnostics["diagnostic_flag"] = ""
diagnostics.loc[~diagnostics["converged"].astype(bool), "diagnostic_flag"] = "MODEL_CONVERGENCE_WARNING"
diagnostics = pd.concat([
    diagnostics,
    pd.DataFrame([
        {
            "model_id": "BOOTSTRAP_FAST_GENERATED_REGRESSOR",
            "outcome": "ALL",
            "grade_signal": "RAW_A|OOF_RESIDUAL_FULL",
            "model_family": "ols_fast_bootstrap",
            "status": "READY_WITH_WARNINGS",
            "converged": True,
            "warning": "Bootstrap regenerates P3 residual inside each replicate but uses fixed development encoder and numpy least-squares for speed.",
            "rank": np.nan,
            "diagnostic_flag": "BOOTSTRAP_APPROXIMATION_WARNING",
        }
    ])
], ignore_index=True)
diagnostics.to_csv(OUTPUT_ROOT / "qa/P4_MODEL_DIAGNOSTICS.csv", index=False)
display(equivalence)
display(diagnostics)

,source,metric,max_abs_raw_minus_residual,allclose,reason
0,end_to_end_cv,iv_deviance,2.282073e-10,False,Residual equals raw A minus fitted P3 controls...
1,end_to_end_cv,iv_brier,3.377730e-11,True,Residual equals raw A minus fitted P3 controls...
2,end_to_end_cv,iv_mae,8.366750e-11,True,Residual equals raw A minus fitted P3 controls...
3,bootstrap_fast_generated_regressor,ame,7.581795e-10,True,Residual equals raw A minus fitted P3 controls...


,model_id,outcome,grade_signal,model_family,status,converged,warning,rank,diagnostic_flag
0,HEALTH_EMPLOYMENT_RAW_A_fractional_logit,HEALTH_EMPLOYMENT,RAW_A,fractional_logit,READY,True,,37.0,
1,HEALTH_EMPLOYMENT_RAW_A_ols_cluster,HEALTH_EMPLOYMENT,RAW_A,ols_cluster,READY,True,,37.0,
2,HEALTH_EMPLOYMENT_OOF_RESIDUAL_FULL_fractional...,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,fractional_logit,READY,True,,37.0,
3,HEALTH_EMPLOYMENT_OOF_RESIDUAL_FULL_ols_cluster,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,ols_cluster,READY,True,,37.0,
4,HEALTH_EMPLOYMENT_OOF_RESIDUAL_NOPOLICY_fracti...,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,fractional_logit,READY,True,,37.0,
5,HEALTH_EMPLOYMENT_OOF_RESIDUAL_NOPOLICY_ols_cl...,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,ols_cluster,READY,True,,37.0,
6,GRAD_SCHOOL_PROGRESSION_RAW_A_fractional_logit,GRAD_SCHOOL_PROGRESSION,RAW_A,fractional_logit,READY,True,,39.0,
7,GRAD_SCHOOL_PROGRESSION_RAW_A_ols_cluster,GRAD_SCHOOL_PROGRESSION,RAW_A,ols_cluster,READY,True,,39.0,
8,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_FULL_frac...,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,fractional_logit,READY,True,,39.0,
9,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_FULL_ols_...,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,ols_cluster,READY,True,,39.0,


In [20]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_df = primary.copy()
plot_df["ame_pp"] = plot_df["ame_10pp"] * 100
for i, (_, row) in enumerate(plot_df.iterrows()):
    ax.scatter(row["ame_pp"], i)
    ax.text(row["ame_pp"], i, f" {row['outcome']} {row['grade_signal']}", va="center", fontsize=8)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("AME for 10pp grade signal, percentage point")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "figures/P4_AME_SIGNAL_SUMMARY.png", dpi=160)
plt.close(fig)

In [21]:
handoff = {
    "created_at": lib.datetime.now(lib.timezone.utc).isoformat(),
    "p3_full_residual_path": lib.rel(p3_full_path),
    "p3_full_residual_sha256": lib.sha256_file(p3_full_path),
    "p3_nopolicy_residual_path": lib.rel(p3_nopolicy_path),
    "p3_nopolicy_residual_sha256": lib.sha256_file(p3_nopolicy_path),
    "recommended_p5_signals": ["RAW_A", "OOF_RESIDUAL_FULL"],
}
(OUTPUT_ROOT / "artifacts/P4_TO_P5_RESIDUAL_HANDOFF.json").write_text(json.dumps(handoff, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([handoff]))

,created_at,p3_full_residual_path,p3_full_residual_sha256,p3_nopolicy_residual_path,p3_nopolicy_residual_sha256,recommended_p5_signals
0,2026-07-13T07:28:51.335476+00:00,workbook/p2/p2_4/p3_grade_residual_v1_strict/d...,d8decd39dca42ccd0dc194fa3813ba0036541098eea08d...,workbook/p2/p2_4/p3_grade_residual_v1_strict/d...,dc5675d941b17108997e63adb550f6f223911892085cf4...,"[RAW_A, OOF_RESIDUAL_FULL]"


In [22]:
bootstrap_success = int(boot.loc[boot["status"].eq("OK")].groupby(["iteration", "grade_signal"]).size().reset_index()["iteration"].nunique())
status = {
    **ENV,
    "P4_INPUT_STATUS": "READY",
    "P4_RAW_A_STATUS": "READY" if "RAW_A" in set(coef["grade_signal"]) else "BLOCKED_SAMPLE",
    "P4_RESIDUAL_STATUS": "READY" if "OOF_RESIDUAL_FULL" in set(coef["grade_signal"]) else "BLOCKED_UPSTREAM_RESIDUAL",
    "P4_EMPLOYMENT_STATUS": "READY" if emp_results["status"].eq("READY").any() else "MODEL_CONVERGENCE_WARNING",
    "P4_PROGRESSION_STATUS": "READY" if prog_results["status"].eq("READY").any() else "MODEL_CONVERGENCE_WARNING",
    "P4_JOINT_DIFFERENCE_STATUS": "READY" if (OUTPUT_ROOT / "artifacts/P4_EMPLOYMENT_PROGRESSION_DIFFERENCE.csv").exists() else "BLOCKED_SAMPLE",
    "P4_BOOTSTRAP_STATUS": "READY" if bootstrap_success >= 500 else "BOOTSTRAP_INCOMPLETE",
    "P4_TEST_STATUS": "READY",
    "P4_TO_P5_STATUS": "READY",
    "P4_OVERALL_STATUS": "READY_WITH_WARNINGS",
    "strict_d08_sha256": lib.sha256_file(INPUTS["strict_d08"]),
    "p2_handoff_sha256": lib.sha256_file(INPUTS["p2_handoff"]),
    "p3_full_residual_sha256": lib.sha256_file(p3_full_path),
}
(OUTPUT_ROOT / "reports/P4_GRADE_SIGNAL_OUTCOME_STATUS.json").write_text(json.dumps(status, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
report = [
    "# P4 Strict Grade Signal Outcome Report",
    "",
    "RAW_A and P3 FULL residual were compared for health employment and graduate-school progression.",
    "",
    "## Coefficients",
    coef.to_string(index=False),
    "",
    "## Incremental value",
    incremental.to_string(index=False),
    "",
    "## Bootstrap CI",
    boot_ci.to_string(index=False),
]
(OUTPUT_ROOT / "reports/P4_GRADE_SIGNAL_OUTCOME_REPORT.md").write_text("\n".join(report), encoding="utf-8")
display(pd.DataFrame([status]))

,python,platform,pandas,numpy,statsmodels,working_directory,git_commit,execution_timestamp_utc,notebook_path,output_root,...,P4_EMPLOYMENT_STATUS,P4_PROGRESSION_STATUS,P4_JOINT_DIFFERENCE_STATUS,P4_BOOTSTRAP_STATUS,P4_TEST_STATUS,P4_TO_P5_STATUS,P4_OVERALL_STATUS,strict_d08_sha256,p2_handoff_sha256,p3_full_residual_sha256
0,3.12.3,Linux-6.18.33.2-microsoft-standard-WSL2-x86_64...,3.0.3,2.5.0,0.14.6,/home/sieg/projects-wsl/SBS_dataScience,5b1a3d54266d881a839ad9a3cec750da66e94bc7,2026-07-13T07:28:29.228482+00:00,/home/sieg/projects-wsl/SBS_dataScience/workbo...,/home/sieg/projects-wsl/SBS_dataScience/workbo...,...,READY,READY,READY,READY,READY,READY,READY_WITH_WARNINGS,5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...,2bbc7d64784b7b530d57bb6a2096d14cb11c5879f41a52...,d8decd39dca42ccd0dc194fa3813ba0036541098eea08d...


In [23]:
manifest = lib.write_manifest(
    OUTPUT_ROOT,
    "P4_OUTPUT_MANIFEST.json",
    NOTEBOOK_PATH,
    {**INPUTS, "p3_full_residual": p3_full_path, "p3_nopolicy_residual": p3_nopolicy_path, "p4_handoff": OUTPUT_ROOT / "artifacts/P4_TO_P5_RESIDUAL_HANDOFF.json"},
    extra={"status_path": lib.rel(OUTPUT_ROOT / "reports/P4_GRADE_SIGNAL_OUTCOME_STATUS.json")},
)
print("manifest:", lib.rel(OUTPUT_ROOT / "logs/P4_OUTPUT_MANIFEST.json"))
print("outputs:", len(manifest["outputs"]))

manifest: workbook/p2/p2_4/p4_grade_signal_outcomes_v1_strict/logs/P4_OUTPUT_MANIFEST.json
outputs: 27


In [24]:
p3_status = json.loads((lib.P3_ROOT / "reports/P3_GRADE_RESIDUAL_STATUS.json").read_text(encoding="utf-8"))
p4_status = json.loads((OUTPUT_ROOT / "reports/P4_GRADE_SIGNAL_OUTCOME_STATUS.json").read_text(encoding="utf-8"))
integrated = {
    "created_at": lib.datetime.now(lib.timezone.utc).isoformat(),
    "P3": {k: p3_status.get(k) for k in ["P3_INPUT_STATUS", "P3_S_FULL_STATUS", "P3_S_NOPOLICY_STATUS", "P3_Q_STATUS", "P3_OOF_COVERAGE_STATUS", "P3_TEST_STATUS", "P3_OVERALL_STATUS"]},
    "P4": {k: p4_status.get(k) for k in ["P4_INPUT_STATUS", "P4_RAW_A_STATUS", "P4_RESIDUAL_STATUS", "P4_EMPLOYMENT_STATUS", "P4_PROGRESSION_STATUS", "P4_JOINT_DIFFERENCE_STATUS", "P4_BOOTSTRAP_STATUS", "P4_TEST_STATUS", "P4_TO_P5_STATUS", "P4_OVERALL_STATUS"]},
    "strict_d08_sha256": p4_status.get("strict_d08_sha256"),
    "p2_handoff_sha256": p4_status.get("p2_handoff_sha256"),
}
integrated_path = PROJECT_ROOT / "workbook/p2/p2_4/P3_P4_CONFIRMATORY_STATUS.json"
integrated_path.write_text(json.dumps(integrated, ensure_ascii=False, indent=2), encoding="utf-8")
md_lines = [
    "# P3/P4 Confirmatory Status",
    "",
    "## P3",
    json.dumps(integrated["P3"], ensure_ascii=False, indent=2),
    "",
    "## P4",
    json.dumps(integrated["P4"], ensure_ascii=False, indent=2),
    "",
    f"strict D08 SHA256: `{integrated['strict_d08_sha256']}`",
    f"P2 handoff SHA256: `{integrated['p2_handoff_sha256']}`",
]
(PROJECT_ROOT / "workbook/p2/p2_4/P3_P4_CONFIRMATORY_STATUS.md").write_text("\n".join(md_lines), encoding="utf-8")
print(lib.rel(integrated_path))

workbook/p2/p2_4/P3_P4_CONFIRMATORY_STATUS.json
